# 柳小乐 v5.1 — Base 预训练（MiniMind 100% · Cosine LR · 续跑）

294M Dense · StdAttn + GQA · QK-Norm · AdamW + Cosine Decay

**AMD MI300X 192GB · BF16**

**v5.1 核心变化：**
- 清洗后 MiniMind pretrain 数据（去 AI 身份/代码噪声/纯英文/外语/翻译）
- MiniMind 8K 分词器（修复 `：`→`<unk>` bug，fertility 0.699）
- 单数据源顺序读取，无需加权采样
- **断点续跑：每次重启从上次数据位置继续，不重复读取**
- **全链路身份清洗：Base → SFT 全程贯穿柳小乐身份**

In [ ]:
# Cell 1：环境 + 核心防护

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

!pip install tokenizers -q 2>&1 | tail -1

import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import time
import random
import math
import json
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"ROCm: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) or 'MI300X'}")
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"显存总量: {mem:.1f} GB")

def print_memory(prefix=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1024**3
        r = torch.cuda.memory_reserved() / 1024**3
        f = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"{prefix}显存: {a:.2f}GB 已用 / {r:.2f}GB 预留 / {f:.2f}GB 空闲")

print_memory("初始")
print("Cell 1 OK")

In [ ]:
# Cell 2：模型配置 + 超参（v5.1 · 清洗数据 · Cosine LR · 续跑）

BASE = Path("/mnt/workspace/shayler2.0")
TOKENIZER = BASE / "tokenizer_minimind_8k/tokenizer.json"
CKPT_DIR = BASE / "checkpoints_v51"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = BASE / "base_data/pretrain_pt_cleaned_v2"  # ★ Cell 3.5 最终清洗 → 分词

from tokenizers import Tokenizer as TokReader
tok = TokReader.from_file(str(TOKENIZER))
VOCAB_SIZE = tok.get_vocab_size()
print(f"分词器词表: {VOCAB_SIZE}")

CONFIG = {
    "vocab_size": VOCAB_SIZE,     # 8000
    "n_layer": 20,
    "n_head": 16,
    "n_query_groups": 4,
    "n_embd": 1024,
    "intermediate_size": 3584,
    "block_size": 1024,
    "norm_eps": 1e-5,
    "diff_attention": False,
    "qk_norm": True,
}

TC = {
    "max_steps": 15000,           # 1,860,181 ÷ 128 ≈ 14,533 → 15000（≈1 epoch）
    "warmup": 500,
    "min_lr_ratio": 0.1,

    "micro_batch_size": 128,
    "grad_accum": 1,

    "lr": 3e-4,
    "weight_decay": 0.01,

    "max_time": int(7.9 * 3600),  # 云平台 8h 限制，续跑兜底
    "save_interval": 100,

    "empty_cache_steps": 200,
    "gc_collect": True,
}

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('medium')  # AMD MI300X 无 TF32，medium 最优

print(f"模型: {CONFIG['n_layer']}层 × {CONFIG['n_embd']}维 ≈ 294M")
print(f"Batch: {TC['micro_batch_size']} | 步数: {TC['max_steps']} | lr={TC['lr']}")
print(f"v5.1：MiniMind 100% · 清洗数据 · 单数据源顺序读取 · Cosine LR decay")
print(f"warmup={TC['warmup']} → cosine → min_lr={TC['lr']*TC['min_lr_ratio']:.1e}")
print(f"Session 限制: {TC['max_time']/3600:.1f}h · save 间隔: {TC['save_interval']}步")
print(f"★ StdAttn + GQA · 标准 AdamW · 同步计时")
print("Cell 2 OK")

In [ ]:
# Cell 3：模型架构（StdAttn + GQA + QK-Norm + SwiGLU）

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight


def apply_rope(x, cos, sin):
    # cos/sin: [1, 1, T, hd//2]，x: [B, nh, T, hd]
    r = x.float().reshape(*x.shape[:-1], -1, 2)
    out0 = r[..., 0] * cos - r[..., 1] * sin
    out1 = r[..., 1] * cos + r[..., 0] * sin
    return torch.stack([out0, out1], dim=-1).flatten(-2).to(x.dtype)


class StdAttn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.nh = c["n_head"]
        self.hd = c["n_embd"] // c["n_head"]
        self.ng = c.get("n_query_groups", self.nh)
        self.scale = self.hd ** -0.5
        kv_dim = self.hd * self.ng
        self.q_proj = nn.Linear(c["n_embd"], c["n_embd"], bias=False)
        self.k_proj = nn.Linear(c["n_embd"], kv_dim, bias=False)
        self.v_proj = nn.Linear(c["n_embd"], kv_dim, bias=False)
        self.o_proj = nn.Linear(c["n_embd"], c["n_embd"], bias=False)
        if c.get("qk_norm", False):
            self.q_norm = RMSNorm(self.hd)
            self.k_norm = RMSNorm(self.hd)
        else:
            self.q_norm = nn.Identity()
            self.k_norm = nn.Identity()

    def forward(self, x, cos, sin, mask=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.nh, self.hd)
        k = self.k_proj(x).view(B, T, self.ng, self.hd)
        v = self.v_proj(x).view(B, T, self.ng, self.hd)
        rp = self.nh // self.ng
        if rp > 1:
            k = k.repeat_interleave(rp, dim=2)
            v = v.repeat_interleave(rp, dim=2)
        q = self.q_norm(q)
        k = self.k_norm(k)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        A = (q @ k.transpose(-2, -1)) * self.scale
        if mask is not None:
            A = A + mask
        A = F.softmax(A, dim=-1)
        out = A @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c["n_embd"]; i = c["intermediate_size"]
        self.w1 = nn.Linear(d, i, bias=False)
        self.w2 = nn.Linear(d, i, bias=False)
        self.w3 = nn.Linear(i, d, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.attn = StdAttn(c)
        self.ffn_norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.ffn = SwiGLU(c)

    def forward(self, x, cos, sin, mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class Shayler(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c; self.bs = c["block_size"]
        self.emb = nn.Embedding(c["vocab_size"], c["n_embd"])
        self.layers = nn.ModuleList([Block(c) for _ in range(c["n_layer"])])
        self.norm = RMSNorm(c["n_embd"], c["norm_eps"])
        self.head = nn.Linear(c["n_embd"], c["vocab_size"], bias=True)
        self.emb.weight = self.head.weight
        self.ckpt_group = 2  # 20层→10组，每组2层（v30 原始设定）

        freqs = 1.0 / (10000 ** (torch.arange(0, c["n_embd"] // c["n_head"], 2).float() / (c["n_embd"] // c["n_head"])))
        t = torch.arange(self.bs).float()
        a = torch.outer(t, freqs)
        self.register_buffer("rc", torch.cos(a).unsqueeze(0).unsqueeze(0))
        self.register_buffer("rs", torch.sin(a).unsqueeze(0).unsqueeze(0))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.emb(idx)
        cos = self.rc[:, :, :T].to(x.device)
        sin = self.rs[:, :, :T].to(x.device)
        mask = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), diagonal=1)

        for g in range(0, len(self.layers), self.ckpt_group):
            def run_group(x, start=g, end=min(g + self.ckpt_group, len(self.layers))):
                for j in range(start, end):
                    x = self.layers[j](x, cos, sin, mask)
                return x
            x = torch.utils.checkpoint.checkpoint(run_group, x, use_reentrant=False)

        x = self.norm(x)
        logits = self.head(x)

        if targets is not None:
            return F.cross_entropy(logits.view(-1, self.cfg["vocab_size"]), targets.view(-1))
        return logits

print("Creating model...")
model = Shayler(CONFIG)
print(f"参数: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print("架构: StdAttn + GQA + QK-Norm + SwiGLU + Pre-Norm RMSNorm")
print(f"★ Gradient checkpointing: ckpt_group=2（10组×2层）")
print("Cell 3 OK")

In [ ]:
# Cell 4：优化器 + LR Schedule 定义
# 注：optimizer 实例在 Cell 6 中创建（需要先迁移模型到 GPU）

print(f"优化器: AdamW (lr={TC['lr']}, wd={TC['weight_decay']}, betas=(0.9, 0.95))")
print(f"LR schedule: warmup {TC['warmup']} 步 → cosine decay → {TC['lr']*TC['min_lr_ratio']:.1e}")
print(f"梯度裁剪: max_norm=1.0")
print("Cell 4 OK")

In [ ]:
# Cell 5：数据加载器（v31 · 单数据源 · 顺序读取 · 断点续跑）

class PretrainDataset(torch.utils.data.IterableDataset):
    """v31 单数据源顺序读取，支持断点续跑。

    续跑机制：
    - state_dict() 保存当前文件索引 + 缓冲区剩余 token
    - load_state_dict() 恢复状态，下次 __iter__ 从断点继续
    - 文件顺序首次随机打乱并持久化到 data_order.json
    - 所有文件读完自动循环，不会停
    - token 总数首次扫描 → 缓存到 data_meta.json，后续秒读
    """

    def __init__(self, data_dir, seq_len, state_dir=None):
        self.data_dir = Path(data_dir)
        self.files = sorted(self.data_dir.glob("*.pt"))
        if not self.files:
            raise FileNotFoundError(f"无 .pt 文件: {data_dir}")

        self.L = seq_len
        self.state_dir = Path(state_dir) if state_dir else None

        # —— 文件顺序持久化（首次随机，之后固定）——
        if self.state_dir:
            self.state_dir.mkdir(parents=True, exist_ok=True)
            order_file = self.state_dir / "data_order.json"
            if order_file.exists():
                with open(order_file) as f:
                    saved = json.load(f)
                saved_set = set(saved)
                current_set = {str(p) for p in self.files}
                ordered = [Path(p) for p in saved if p in current_set]
                new = [p for p in self.files if str(p) not in saved_set]
                self.files = ordered + new
                print(f"  📋 文件顺序已恢复 ({len(self.files)} 文件)")
            else:
                random.shuffle(self.files)
                with open(order_file, 'w') as f:
                    json.dump([str(p) for p in self.files], f)
                print(f"  🎲 文件顺序已随机化并保存 ({len(self.files)} 文件)")
        else:
            random.shuffle(self.files)

        # —— 状态 ——
        self._file_idx = 0      # 当前正在读取的文件索引
        self._buf = torch.empty(0, dtype=torch.long)  # 缓冲区（剩余 token）

        # 统计总 token（首次扫描 → 缓存，后续秒读）
        meta_file = self.data_dir / "data_meta.json"
        if meta_file.exists():
            with open(meta_file) as f:
                meta = json.load(f)
            total_tokens = meta.get("total_tokens", 0)
            print(f"  📊 缓存命中: {total_tokens/1e9:.2f}B tokens")
        else:
            print(f"  🔍 首次扫描 ({len(self.files)} 文件)...")
            total_tokens = 0
            for f in self.files:
                t = torch.load(f, map_location="cpu", weights_only=True)
                total_tokens += t.numel()
            with open(meta_file, 'w') as f:
                json.dump({"total_tokens": total_tokens, "num_files": len(self.files)}, f)
            print(f"  ✅ 扫描完成: {total_tokens/1e9:.2f}B tokens, 已缓存到 data_meta.json")

        self.total_tokens = total_tokens
        print(f"Dataset: {len(self.files)} 文件, {total_tokens/1e9:.2f}B tokens")

    def state_dict(self):
        """保存当前读取位置（checkpoint 时调用）"""
        return {
            'file_idx': self._file_idx,
            'buf': self._buf.clone(),
        }

    def load_state_dict(self, sd):
        """恢复读取位置（续跑时调用）"""
        self._file_idx = sd.get('file_idx', 0)
        self._buf = sd.get('buf', torch.empty(0, dtype=torch.long))
        # 防止 file_idx 越界（文件被删除的情况）
        if self._file_idx >= len(self.files):
            self._file_idx = self._file_idx % len(self.files)
        print(f"  📂 数据续跑: file_idx={self._file_idx}/{len(self.files)}, "
              f"buf_len={len(self._buf):,} tokens")

    def __iter__(self):
        file_idx = self._file_idx
        buf = self._buf.clone()

        if file_idx == 0 and len(buf) == 0:
            print(f"  🆕 数据迭代器从零开始")
        else:
            print(f"  ▶ 续跑: file_idx={file_idx}, buf={len(buf):,} tokens")

        while True:
            # 确保缓冲区有足够 token
            while len(buf) < self.L + 1:
                if file_idx >= len(self.files):
                    print(f"  🔄 Epoch 完成，回到文件 0")
                    file_idx = 0

                ids = torch.load(self.files[file_idx], map_location="cpu", weights_only=True).long()
                buf = torch.cat([buf, ids.view(-1)])  # 展平 [N, 1024] → 1D
                file_idx += 1

            # 取出一段
            chunk = buf[:self.L + 1]
            buf = buf[self.L:]

            # ★ 更新实例状态（yield 前保存，确保 checkpoint 拿到正确位置）
            self._file_idx = file_idx
            self._buf = buf.clone()

            yield chunk[:-1], chunk[1:]  # (input, target)


print("Data Loader OK（v31 · 单数据源 · 断点续跑）")
print("Cell 5 OK")

In [ ]:
# Cell 6：GPU 迁移 + 训练循环（v5.1 · 续跑 · Cosine LR）

# ———— 数据检查 ————
print("\n数据检查:")
if DATA_DIR.exists():
    n = len(list(DATA_DIR.glob("*.pt")))
    print(f"  {DATA_DIR.name}: {n} 个 .pt 文件 ✓")
else:
    print(f"  ⚠️ {DATA_DIR} 不存在！请先运行 clean_base_pretrain_data.ipynb")

# ———— 迁移模型到 GPU ————
print("\n迁移模型到 GPU...")
model.cuda().train()
print_memory("模型迁移后")

# ———— 续跑检测 ————
latest = None
start_step = 0
for ckpt in sorted(CKPT_DIR.glob("step_*.pt")):
    try:
        s = int(ckpt.stem.split("_")[1])
        if s > start_step:
            start_step = s
            latest = ckpt
    except Exception:
        pass

device = torch.cuda.current_device()
lema = None
data_state = None

if latest:
    print(f"\n{'='*50}")
    print(f"📂 续跑: {latest.name}")
    sd = torch.load(latest, map_location="cpu")
    model.load_state_dict(sd["model"])
    lema = sd.get("loss_ema", None)
    start_step = sd.get("step", start_step)
    data_state = sd.get("data_state", None)
    if lema:
        print(f"   step={start_step}  loss_ema={lema:.4f}")
    else:
        print(f"   step={start_step}")

    opt = torch.optim.AdamW(model.parameters(), lr=TC['lr'],
                            betas=(0.9, 0.95), weight_decay=TC['weight_decay'])
    if "optimizer" in sd:
        try:
            opt.load_state_dict(sd["optimizer"])
            print("   ✅ 优化器状态已恢复")
        except Exception:
            print("   ⚠️ 优化器状态不兼容，重新初始化")
        for pg in opt.param_groups:
            pg["lr"] = TC["lr"]
            pg["weight_decay"] = TC["weight_decay"]

    sps = 6.6
    remaining = TC['max_steps'] - start_step
    sessions_needed = (remaining * sps) / TC['max_time']
    print(f"   剩余: {remaining} 步 → ~{remaining * sps / 3600:.0f}h（约 {sessions_needed:.0f} 次 session）")
    del sd
    torch.cuda.empty_cache()
    print(f"{'='*50}")
else:
    print("\n🆕 v5.1 Base 预训练启动：MiniMind 100% · 清洗数据")
    print(f"   模型: 294M | 数据: pretrain_pt_cleaned | 分词器: MiniMind 8K")
    print(f"   Cosine LR: warmup={TC['warmup']} → min_lr={TC['lr']*TC['min_lr_ratio']:.1e}")
    opt = torch.optim.AdamW(model.parameters(), lr=TC['lr'],
                            betas=(0.9, 0.95), weight_decay=TC['weight_decay'])

# ———— DataLoader ————
print("\n初始化 DataLoader...")
ds = PretrainDataset(DATA_DIR, CONFIG["block_size"], state_dir=CKPT_DIR)

if data_state is not None:
    ds.load_state_dict(data_state)

dl = iter(torch.utils.data.DataLoader(
    ds, batch_size=TC["micro_batch_size"],
    num_workers=0, pin_memory=False))
print("DataLoader 就绪")

# 动态计算 steps/epoch
blocks_per_step = TC["micro_batch_size"] * TC["grad_accum"]
steps_per_epoch = ds.total_tokens // (CONFIG["block_size"] * blocks_per_step)
print(f"数据配置: {ds.total_tokens/1e9:.2f}B tokens → 1 epoch ≈ {steps_per_epoch:,} steps")
print(f"         {TC['max_steps']:,} steps ≈ {TC['max_steps']/steps_per_epoch:.1f} epochs")

# ═══════════════════════════════════════════════
# 训练循环
# ═══════════════════════════════════════════════
total = TC["max_steps"]
warmup = TC["warmup"]
max_t = TC["max_time"]
min_lr_ratio = TC["min_lr_ratio"]
tokens_per_step = TC["micro_batch_size"] * CONFIG["block_size"]  # 128 × 1024 = 131K

print(f"\n{'='*50}")
print(f"训练: step {start_step} → {total}")
print(f"数据: MiniMind pretrain 100% · 清洗数据 · MiniMind 8K 分词器")
print(f"优化器: AdamW lr={TC['lr']} wd={TC['weight_decay']}")
print(f"LR: warmup {warmup}步 → cosine → {TC['lr']*min_lr_ratio:.1e}")
print(f"Session 限制: {max_t/3600:.1f}h | 保存间隔: {TC['save_interval']}步 | 保留最近5个")
print(f"{'='*50}")

# ★ Loss 日志
LOSS_LOG = CKPT_DIR / "loss_log.csv"
_loss_log_first_write = not LOSS_LOG.exists() or LOSS_LOG.stat().st_size == 0
_loss_file = open(LOSS_LOG, 'a')
if _loss_log_first_write:
    _loss_file.write("step,loss,ema,lr,grad_norm\n")
    print(f"📋 新建 loss 日志: {LOSS_LOG}")
else:
    print(f"📋 续写 loss 日志: {LOSS_LOG}")

step = start_step
t0 = time.time()
t_data = t_fwd = t_bwd = t_opt = 0.0

while step < total:
    elapsed = time.time() - t0
    if elapsed > max_t:
        print(f"\n⏰ {elapsed/3600:.1f}h 到达 session 时限 ({max_t/3600:.1f}h)，保存退出...")
        break

    # ═══════════════════════════════════════════
    # Cosine LR schedule
    # ═══════════════════════════════════════════
    if step < warmup:
        lr_scale = step / max(warmup, 1)
    else:
        progress = (step - warmup) / max(total - warmup, 1)
        lr_scale = min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    current_lr = TC["lr"] * lr_scale

    for pg in opt.param_groups:
        pg["lr"] = current_lr

    accum_loss = 0
    opt.zero_grad(set_to_none=True)

    for acc_step in range(TC["grad_accum"]):
        torch.cuda.synchronize()
        _t0 = time.time()
        x, y = next(dl)
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        torch.cuda.synchronize()
        t_data += time.time() - _t0

        torch.cuda.synchronize()
        _t0 = time.time()
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            loss = model(x, y)
        torch.cuda.synchronize()
        t_fwd += time.time() - _t0

        torch.cuda.synchronize()
        _t0 = time.time()
        (loss / TC["grad_accum"]).backward()
        torch.cuda.synchronize()
        t_bwd += time.time() - _t0

        accum_loss += loss.item()
        del loss

    torch.cuda.synchronize()
    _t0 = time.time()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    torch.cuda.synchronize()
    t_opt += time.time() - _t0

    if step % TC["empty_cache_steps"] == 0 and step > 0:
        torch.cuda.empty_cache()
        if TC["gc_collect"]:
            gc.collect(generation=2)
        torch.cuda.reset_peak_memory_stats()

    step += 1
    avg_loss = accum_loss / TC["grad_accum"]
    if lema is None:
        lema = avg_loss
    else:
        lema = 0.95 * lema + 0.05 * avg_loss

    # 每步写 loss 日志
    _loss_file.write(f"{step},{avg_loss:.6f},{lema:.6f},{current_lr:.8f},{grad_norm:.4f}\n")
    _loss_file.flush()

    if step % 10 == 0:
        eh = (time.time() - t0) / 3600
        n_steps = step - start_step
        sps = n_steps / (time.time() - t0) if time.time() > t0 else 0
        rh = (total - step) / sps / 3600 if sps > 0 else 0
        mem_alloc = torch.cuda.memory_allocated() / 1024**3
        lr_tag = "[WARMUP]" if step <= warmup else "[COSINE]"
        total_t = t_data + t_fwd + t_bwd + t_opt
        avg_t = total_t / n_steps if n_steps > 0 else 0
        tok_per_sec = tokens_per_step / avg_t if avg_t > 0 else 0
        print(f"  {step}/{total}({step/total*100:.1f}%) loss={avg_loss:.4f} ema={lema:.4f} "
              f"{lr_tag} lr={current_lr:.1e} gn={grad_norm:.1f} "
              f"⏱{avg_t:.1f}s/步 {tok_per_sec/1000:.0f}K tok/s | "
              f"{eh:.1f}h 剩~{rh:.0f}h | Mem: {mem_alloc:.2f}GB")
        if n_steps > 0:
            print(f"    ⏱ 每步: data={t_data/n_steps*1000:.0f}ms fwd={t_fwd/n_steps*1000:.0f}ms bwd={t_bwd/n_steps*1000:.0f}ms opt={t_opt/n_steps*1000:.0f}ms")

    if step % TC["save_interval"] == 0:
        ckpt_path = CKPT_DIR / f"step_{step:06d}.pt"
        torch.save({
            "step": step,
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "loss_ema": lema,
            "config": CONFIG,
            "data_state": ds.state_dict(),
        }, ckpt_path)
        # 保留最近 5 个 checkpoint
        all_ckpts = sorted(CKPT_DIR.glob("step_*.pt"), key=lambda p: p.stat().st_mtime)
        for old in all_ckpts[:-5]:
            old.unlink()
            print(f"  🗑️ 删除旧ckpt: {old.name}")
        print(f"  💾 {ckpt_path.name} | step={step} ema={lema:.4f} lr={current_lr:.1e}")

# ———— 最终保存 ————
if step > start_step and step % TC["save_interval"] != 0:
    ckpt_path = CKPT_DIR / f"step_{step:06d}.pt"
    torch.save({
        "step": step,
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "loss_ema": lema,
        "config": CONFIG,
        "data_state": ds.state_dict(),
    }, ckpt_path)
    print(f"\n💾 最终保存: {ckpt_path.name} | step={step} ema={lema:.4f}")

_loss_file.close()
print(f"📋 loss 日志已关闭: {LOSS_LOG}")

if step >= total:
    print(f"\n🎉 v5.1 Base 预训练完成！")
    print(f"   终局: step={step} ema={lema:.4f} gn={grad_norm:.1f} lr={current_lr:.1e}")
else:
    print(f"\n⏸ Session 结束 | step={step}/{total} ({step/total*100:.1f}%)")
    remaining = total - step
    sps = (step - start_step) / (time.time() - t0) if time.time() > t0 else 6.6
    sessions_left = (remaining * sps) / TC['max_time'] if sps > 0 else 0
    if sps > 0:
        print(f"   剩余: {remaining} 步 → ~{remaining * sps / 3600:.0f}h（约 {sessions_left:.0f} 次 session）")
    print(f"   下次续跑: 重新运行本 Cell 即可自动恢复")
    print(f"   ⚠️ 注意: 重启后直接 Run All，checkpoint 会自动检测并恢复")

In [ ]:
# Cell 7：进度查看

ckpts = sorted(CKPT_DIR.glob("step_*.pt"))
pt_files = sorted(DATA_DIR.glob("*.pt"))
if ckpts:
    print(f"{len(ckpts)} 个 checkpoint ({len(pt_files)} 个数据文件):")
    for c in ckpts:
        s = torch.load(c, map_location='cpu')
        mb = c.stat().st_size / 1e6
        st = s.get('step', 0)
        le = s.get('loss_ema', 0)
        pct = st / TC['max_steps'] * 100
        ds_state = s.get('data_state', {})
        fi = ds_state.get('file_idx', '?')
        print(f"  {c.name:20s} step={st:>6} ({pct:.1f}%) ema={le:.4f}  data_file={fi}  {mb:.0f}MB")
else:
    print("尚无 checkpoint")

print(f"\n目标: {TC['max_steps']} steps | v5.1 MiniMind 100% · 清洗数据")
print(f"数据: {len(pt_files)} 个 .pt 文件 | 分词器: MiniMind 8K")

# 数据循环进度
if ckpts:
    latest_sd = torch.load(ckpts[-1], map_location='cpu')
    ds_state = latest_sd.get('data_state', {})
    if ds_state:
        fi = ds_state.get('file_idx', 0)
        print(f"\n📂 数据进度: 当前在第 {fi}/{len(pt_files)} 个文件")
        if fi > 0:
            print(f"   已完成 ~{fi/len(pt_files)*100:.0f}% → 约 {fi/len(pt_files):.1f} epoch")